# Academic Research Workflow: Full Analysis Pipeline

This notebook is designed for **graduate students and researchers** who need
publication-quality beta estimation with full diagnostics, benchmark comparisons,
and validation checks.

We will walk through the complete research pipeline:

1. Research-grade estimation with tuned hyperparameters
2. Inspecting the fitted model
3. Evaluation metrics
4. Benchmark comparison against rolling OLS
5. Lookahead bias validation
6. Exporting results for publication

In [ ]:
!pip install grubeta[full] -q

## Step 1: Research-Grade Estimation

The `preset="research"` option configures the model for maximum accuracy:
longer training, finer learning rate schedule, and full convergence checks.

In [ ]:
from grubeta import estimate_beta

result = estimate_beta("AAPL", "SPY", preset="research")
print(result["summary"])

## Step 2: Access the Fitted Model

The result dictionary exposes the underlying model and its configuration
for reproducibility and inspection.

In [ ]:
# Inspect the model and its configuration
model = result["model"]
print("Model configuration:")
print(model.config)

print("\nResults summary statistics:")
print(result["results"].describe())

## Step 3: Evaluation Metrics

Use `BetaEvaluator` to compute standard evaluation metrics for the
estimated beta series, including in-sample fit and stability diagnostics.

In [ ]:
from grubeta.evaluation import BetaEvaluator

evaluator = BetaEvaluator(result)
metrics = evaluator.evaluate()

print("Evaluation Metrics:")
for key, value in metrics.items():
    print(f"  {key}: {value}")

## Step 4: Benchmark Comparison

Compare the GRU-estimated beta against a standard rolling OLS baseline.
This is essential for demonstrating that the neural approach adds value
over traditional methods.

In [ ]:
from grubeta.benchmarks import rolling_ols_beta
import pandas as pd

# Compute rolling OLS beta as a baseline
ols_beta = rolling_ols_beta(
    result["results"]["asset_returns"],
    result["results"]["market_returns"],
    window=126
)

# Side-by-side comparison
comparison = pd.DataFrame({
    "GRU Beta": result["beta"],
    "Rolling OLS Beta": ols_beta
}).dropna()

print("Comparison statistics:")
print(comparison.describe())

print(f"\nCorrelation between GRU and OLS beta: {comparison.corr().iloc[0, 1]:.4f}")

## Step 5: Lookahead Bias Validation

A critical check for any time-series model: verify that the estimated beta
at time *t* does not use any information from time *t+1* or later.

The `validate_no_lookahead` function runs an expanding-window test to confirm
that predictions are stable when future data is excluded.

In [ ]:
from grubeta.validation import validate_no_lookahead

validation_result = validate_no_lookahead(result)

print(f"Lookahead bias detected: {validation_result['has_lookahead']}")
print(f"Max deviation:           {validation_result['max_deviation']:.6f}")
print(f"Mean deviation:          {validation_result['mean_deviation']:.6f}")

## Step 6: Export for Publication

Save your results to CSV for use in LaTeX tables, or generate a summary report.

```python
# Save the beta series to CSV
result["beta"].to_csv("aapl_dynamic_beta.csv", header=["beta"])

# Save the full results DataFrame
result["results"].to_csv("aapl_full_results.csv")

# Save evaluation metrics
import json
with open("aapl_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2, default=str)
```

For reproducibility, always record:
- The GRUBeta version (`import grubeta; print(grubeta.__version__)`)
- The model configuration (`result["model"].config`)
- The data date range used for estimation